# Advanced Feature: Agent Tool Usage

This notebook demonstrates how `TinyPerson` agents in TinyTroupe can be equipped with "tools" to extend their capabilities beyond basic conversation. Tools allow agents to perform more complex actions, such as creating documents, searching databases, or interacting with external APIs (though this example focuses on document creation).

We will cover:
1. Setting up the necessary imports, including `TinyToolUse` (the mental faculty for tool usage) and a specific tool, `TinyWordProcessor`.
2. Creating agents using `TinyPersonFactory`.
3. Configuring an `ArtifactExporter` to specify where files created by tools (like documents) should be saved.
4. Optionally setting up a `TinyEnricher` for post-processing content created by tools.
5. Creating an instance of the `TinyToolUse` faculty and equipping it with the `TinyWordProcessor` tool.
6. Adding this tool-using faculty to our agents.
7. Simulating a scenario where agents are tasked to brainstorm and then write a document about their ideas.
8. Observing how agents decide to use the `WRITE_DOCUMENT` action, which is provided by the `TinyWordProcessor` tool.
9. Checking the output directory for the document created by an agent.

## 1. Setup and Imports

Import core classes, the `TinyToolUse` faculty, and the `TinyWordProcessor` tool. `ArtifactExporter` is needed to manage where tool-generated files are saved, and `TinyEnricher` can optionally post-process them.

In [ ]:
import json
import sys
# If running from 'examples/advanced_features/', this adds the project root to Python path.
sys.path.append('../..') 

import tinytroupe # Initializes configuration, logging, etc.
from tinytroupe.agent import TinyPerson, TinyToolUse
from tinytroupe.environment import TinyWorld # TinySocialNetwork not used in this example
from tinytroupe.factory import TinyPersonFactory
# extractor and ResultsReducer not directly used in this specific tool usage flow
# from tinytroupe.extraction import default_extractor as extractor
# from tinytroupe.extraction import ResultsReducer 
# import tinytroupe.control as control # Not directly used
from tinytroupe.extraction import ArtifactExporter
from tinytroupe.enrichment import TinyEnricher

from tinytroupe.tools import TinyWordProcessor

## 2. Create Agents

We'll generate a couple of agents who work for a marketing services company. Their roles might involve creating written content.

In [ ]:
factory_context = "A dynamic company providing innovative marketing services to a diverse range of clients."
factory = TinyPersonFactory(factory_context)

print("Generating agents...")
agent_1 = factory.generate_person("Sophia, a Content Strategist who is highly organized and creative.")
agent_2 = factory.generate_person("Liam, a Marketing Analyst who is detail-oriented and enjoys collaborative work.")

if agent_1 and agent_2:
    print(f"Generated agent: {agent_1.name}")
    print(f"Generated agent: {agent_2.name}")
else:
    print("Agent generation failed. Check logs for details.")

## 3. Configure Tool Usage Faculty

To allow agents to use tools, we need to:
1. Set up an `ArtifactExporter`: This tells the tools where to save any files they create (e.g., documents, spreadsheets). We'll set this to save in an `outputs` directory relative to the `examples` folder.
2. Optionally, set up a `TinyEnricher`: This can be used by tools to post-process generated content (e.g., adding metadata, improving formatting). For this example, a default enricher is fine.
3. Create a `TinyToolUse` faculty: This mental faculty is given a list of tool instances (like `TinyWordProcessor`).
4. Add the `TinyToolUse` faculty to the agents.

In [ ]:
# Define where tool-generated artifacts will be saved.
output_directory = "../outputs/tool_usage_example" # Path relative to this notebook's location
exporter = ArtifactExporter(base_output_folder=output_directory)
print(f"Tool artifacts will be saved in: {output_directory}/")

# Enricher can post-process tool outputs (optional for basic use)
enricher = TinyEnricher()

# Create the TinyWordProcessor tool, passing the exporter and enricher
word_processor_tool = TinyWordProcessor(exporter=exporter, enricher=enricher)

# Create the TinyToolUse faculty and equip it with the word processor tool
tool_use_faculty = TinyToolUse(tools=[word_processor_tool])

# Add the tool-using faculty to our agents
if agent_1:
    agent_1.add_mental_faculties([tool_use_faculty])
    print(f"{agent_1.name} is now equipped with TinyWordProcessor.")
if agent_2:
    agent_2.add_mental_faculties([tool_use_faculty])
    print(f"{agent_2.name} is now equipped with TinyWordProcessor.")

### 3.1. (Optional) Inspect Agent Specification for Tool Actions

Adding a tool faculty modifies the agent's underlying prompt, informing it of the new actions it can perform (like `WRITE_DOCUMENT` from `TinyWordProcessor`). You can inspect this by printing the agent's specification to see these new capabilities listed.

In [ ]:
if agent_1:
    print(f"--- Partial Specification for {agent_1.name} showing tool actions ---")
    # The full specification is very long, so we'll just show a part that includes action definitions
    spec_lines = agent_1.generate_agent_system_prompt().splitlines()
    action_section_started = False
    tool_action_details_printed = False
    for line in spec_lines:
        if "You have the following types of actions available to you:" in line:
            action_section_started = True
        if action_section_started and "WRITE_DOCUMENT" in line and not tool_action_details_printed:
            # Try to print the action and its description block
            current_index = spec_lines.index(line)
            for i in range(current_index, min(current_index + 5, len(spec_lines))):
                print(spec_lines[i])
            tool_action_details_printed = True # Print details only once
        if action_section_started and "Whenever you WRITE_DOCUMENT" in line:
            print("\nConstraint for WRITE_DOCUMENT:")
            print(line)
            break # Stop after showing relevant part for tool

## 4. Simulate a Task Requiring Tool Use

We'll create a simple world, add our agents, and then give them a task: to brainstorm ideas and write a document about them. This should prompt them to use the `WRITE_DOCUMENT` action provided by the `TinyWordProcessor`.

In [ ]:
if agent_1 and agent_2:
    company_world = TinyWorld("Marketing Department", [agent_1, agent_2])
    company_world.make_everyone_accessible()
    print(f"\nWorld '{company_world.name}' created with agents: {[agent.name for agent in company_world.agents]}")

    # Task for the agents
    task_prompt = "Hello team! Let's brainstorm one or two innovative marketing campaign ideas for a new eco-friendly coffee brand. After discussing, please have one of you write a short document summarizing the chosen idea(s), its target audience, and key messaging."
    print(f"\nBroadcasting task: {task_prompt}")
    company_world.broadcast(task_prompt)

    # Run the simulation for a few steps to allow discussion and document creation
    # The TinyWordProcessor will automatically save any document written by an agent to the path defined in ArtifactExporter.
    print("\n--- Running simulation... ---")
    # Using run_minutes which is a convenience wrapper around world.run()
    company_world.run_minutes(2) # Allow a couple of minutes (steps) for interaction
    print("\nSimulation finished.")
    print(f"Check the '{output_directory}/' folder for any documents created by the agents.")
    print(f"Expected path might be like: {output_directory}/{company_world.name}/<AgentName>/<AgentName> - <DocumentTitle>_<Timestamp>.md")
else:
    print("Skipping simulation as one or more agents failed to generate.")

## 5. Observing Tool Use and Output

Review the console output above. You should see one or more agents using the `WRITE_DOCUMENT` action. The `TinyWordProcessor`, through the `ArtifactExporter` we configured, will save the content of this action as a Markdown file.

**Navigate to the `examples/outputs/tool_usage_example/` directory (or the specific subdirectory shown in the output above, which includes the world and agent name) to find the document(s) created by the agent(s).**

This example illustrates how tools can seamlessly extend agent capabilities, enabling them to perform more complex, structured tasks like content creation within the simulation, with tangible outputs saved to the filesystem.